# Data exploration & scikit-learn: practice notebook

2 exercises, each with hidden tests. Data is loaded as **polars**; pandas only appears where a library insists on it.

**How to use it**
1. Run the three setup cells below once (Runtime → Run all works too; the exercise cells just print `Wrong answer` until you solve them).
2. For each exercise, write your code in the solution cell, run it, then run the `check(...)` cell under it. It prints **Accepted** or which test failed and why.
3. Stuck? `solution("e1")` prints the reference solution.

Opened from the roadmap it runs in Google Colab. To keep your work, use File → Save a copy in Drive the first time.

In [ ]:
# Install anything missing (Colab already has numpy, scipy, statsmodels, scikit-learn, matplotlib)
import importlib.util, subprocess, sys
for _p in ['polars']:
    if importlib.util.find_spec(_p) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', _p], check=False)


## Setup: data
The data every exercise uses. Read it if you like; don't change it.

In [ ]:
import numpy as np
import polars as pl
import matplotlib.pyplot as plt
_g = np.random.default_rng(2026)
_T = ["AAA", "BBB", "CCC", "DDD"]
_cov = 0.0001 * np.array([[4, 2, 1, 0.5], [2, 3, 1, 0.5], [1, 1, 2, 0.2], [0.5, 0.5, 0.2, 1]])
_r = _g.multivariate_normal([0.0006, 0.0004, 0.0003, 0.0001], _cov, 500)
_r[_g.uniform(size=_r.shape) < 0.01] = np.nan
_days = np.arange(np.datetime64("2024-01-01"), np.datetime64("2026-12-31"))
_days = _days[np.is_busday(_days)][:500]
rets = pl.DataFrame({"date": _days, **{t: _r[:, i] for i, t in enumerate(_T)}}).with_columns(pl.col(_T).fill_nan(None))
_m = 2000
_inc = np.round(_g.lognormal(11, 0.4, _m), -2)
_dr = np.round(_g.beta(2, 5, _m), 3)
_age = _g.integers(21, 70, _m)
_late = _g.poisson(0.6, _m)
_z = -1.0 - 0.00002 * (_inc - 60000) + 3.0 * _dr - 0.02 * (_age - 40) + 0.7 * _late
credit = pl.DataFrame({"income": _inc, "debt_ratio": _dr, "age": _age, "late_payments": _late,
                       "default": (_g.uniform(size=_m) < 1 / (1 + np.exp(-_z))).astype(int)})

In [ ]:
# Test runner: check(id) and solution(id)
import base64 as _b64, json as _json, io as _io, contextlib as _ctx, traceback as _tb
_LAB = _json.loads(_b64.b64decode("eyJlMSI6IHsic29sdXRpb24iOiAiZGVmIHN1bW1hcnkocmV0cyk6XG4gICAgdGlja2VycyA9IFtjIGZvciBjIGluIHJldHMuY29sdW1ucyBpZiBjICE9IFwiZGF0ZVwiXVxuICAgIHJvd3MgPSBbXVxuICAgIGZvciB0IGluIHRpY2tlcnM6XG4gICAgICAgIHIgPSByZXRzW3RdXG4gICAgICAgIHdlYWx0aCA9ICgxICsgci5maWxsX251bGwoMCkpLmN1bV9wcm9kKClcbiAgICAgICAgcm93cy5hcHBlbmQoe1widGlja2VyXCI6IHQsXG4gICAgICAgICAgICAgICAgICAgICBcImFubl9yZXR1cm5cIjogci5tZWFuKCkgKiAyNTIsXG4gICAgICAgICAgICAgICAgICAgICBcImFubl92b2xcIjogci5zdGQoKSAqIG5wLnNxcnQoMjUyKSxcbiAgICAgICAgICAgICAgICAgICAgIFwibWF4X2RyYXdkb3duXCI6ICh3ZWFsdGggLyB3ZWFsdGguY3VtX21heCgpIC0gMSkubWluKCl9KVxuICAgIHJldHVybiAocGwuRGF0YUZyYW1lKHJvd3MpXG4gICAgICAgICAgICAgIC53aXRoX2NvbHVtbnMoKHBsLmNvbChcImFubl9yZXR1cm5cIikgLyBwbC5jb2woXCJhbm5fdm9sXCIpKS5hbGlhcyhcInNoYXJwZVwiKSlcbiAgICAgICAgICAgICAgLnNlbGVjdChcInRpY2tlclwiLCBcImFubl9yZXR1cm5cIiwgXCJhbm5fdm9sXCIsIFwic2hhcnBlXCIsIFwibWF4X2RyYXdkb3duXCIpXG4gICAgICAgICAgICAgIC5zb3J0KFwic2hhcnBlXCIsIGRlc2NlbmRpbmc9VHJ1ZSkpXG5cbmRlZiBwbG90KHJldHMpOlxuICAgIHRpY2tlcnMgPSBbYyBmb3IgYyBpbiByZXRzLmNvbHVtbnMgaWYgYyAhPSBcImRhdGVcIl1cbiAgICBmaWcsIChheDEsIGF4MikgPSBwbHQuc3VicGxvdHMoMSwgMiwgZmlnc2l6ZT0oMTEsIDQpKVxuICAgIHdlYWx0aCA9IHJldHMuc2VsZWN0KFwiZGF0ZVwiLCAqWygxICsgcGwuY29sKHQpLmZpbGxfbnVsbCgwKSkuY3VtX3Byb2QoKS5hbGlhcyh0KSBmb3IgdCBpbiB0aWNrZXJzXSlcbiAgICBmb3IgdCBpbiB0aWNrZXJzOlxuICAgICAgICBheDEucGxvdCh3ZWFsdGhbXCJkYXRlXCJdLCB3ZWFsdGhbdF0sIGxhYmVsPXQpXG4gICAgYXgxLnNldF90aXRsZShcIkN1bXVsYXRpdmUgd2VhbHRoXCIpOyBheDEubGVnZW5kKClcbiAgICBjID0gcmV0cy5kcm9wKFwiZGF0ZVwiKS5kcm9wX251bGxzKCkuY29ycigpLnRvX251bXB5KClcbiAgICBpbSA9IGF4Mi5pbXNob3coYywgdm1pbj0tMSwgdm1heD0xLCBjbWFwPVwiUmRCdV9yXCIpXG4gICAgYXgyLnNldF94dGlja3MocmFuZ2UobGVuKHRpY2tlcnMpKSwgdGlja2Vycyk7IGF4Mi5zZXRfeXRpY2tzKHJhbmdlKGxlbih0aWNrZXJzKSksIHRpY2tlcnMpXG4gICAgYXgyLnNldF90aXRsZShcIkNvcnJlbGF0aW9uXCIpOyBmaWcuY29sb3JiYXIoaW0sIGF4PWF4Mik7IGZpZy50aWdodF9sYXlvdXQoKVxuIiwgInRlc3RzIjogIlxuZGVmIGNsb3NlKGEsIGIsIHJlbD0xZS02LCB3aGF0PVwidmFsdWVcIik6XG4gICAgYXNzZXJ0IGEgaXMgbm90IE5vbmUsIFwieW91ciBmdW5jdGlvbiByZXR1cm5lZCBOb25lLiBEaWQgeW91IGZvcmdldCB0byByZXR1cm4/XCJcbiAgICBhID0gZmxvYXQoYSk7IGIgPSBmbG9hdChiKVxuICAgIGFzc2VydCBhYnMoYSAtIGIpIDw9IHJlbCAqIG1heCgxLjAsIGFicyhiKSksIGZcInt3aGF0fSA9IHthOi42Z30sIGV4cGVjdGVkIHtiOi42Z31cIlxuZGVmIGJldHdlZW4oYSwgbG8sIGhpLCB3aGF0PVwidmFsdWVcIik6XG4gICAgYXNzZXJ0IGEgaXMgbm90IE5vbmUsIFwieW91ciBmdW5jdGlvbiByZXR1cm5lZCBOb25lLiBEaWQgeW91IGZvcmdldCB0byByZXR1cm4/XCJcbiAgICBhc3NlcnQgbG8gPD0gZmxvYXQoYSkgPD0gaGksIGZcInt3aGF0fSA9IHtmbG9hdChhKTouNGZ9LCBleHBlY3RlZCBiZXR3ZWVuIHtsb30gYW5kIHtoaX1cIlxuZGVmIHNhbWVfa2V5cyhhLCBiLCB3aGF0PVwicmVzdWx0XCIpOlxuICAgIGFzc2VydCBpc2luc3RhbmNlKGEsIGRpY3QpLCBmXCJ7d2hhdH0gc2hvdWxkIGJlIGEgZGljdCwgZ290IHt0eXBlKGEpLl9fbmFtZV9ffVwiXG4gICAgYXNzZXJ0IHNldChhKSA9PSBzZXQoYiksIGZcInt3aGF0fSBrZXlzIHtzb3J0ZWQoYSl9LCBleHBlY3RlZCB7c29ydGVkKGIpfVwiXG5cbmRlZiBfdChkZik6XG4gICAgYSA9IFVbXCJzdW1tYXJ5XCJdKGRmKTsgYiA9IFJbXCJzdW1tYXJ5XCJdKGRmKVxuICAgIGFzc2VydCBpc2luc3RhbmNlKGEsIHBsLkRhdGFGcmFtZSksIGZcImV4cGVjdGVkIGEgcG9sYXJzIERhdGFGcmFtZSwgZ290IHt0eXBlKGEpLl9fbmFtZV9ffVwiXG4gICAgYXNzZXJ0IGEuY29sdW1ucyA9PSBiLmNvbHVtbnMsIGZcImNvbHVtbnMge2EuY29sdW1uc30sIGV4cGVjdGVkIHtiLmNvbHVtbnN9XCJcbiAgICBhc3NlcnQgYVtcInRpY2tlclwiXS50b19saXN0KCkgPT0gYltcInRpY2tlclwiXS50b19saXN0KCksIGZcInJvdyBvcmRlciB7YVsndGlja2VyJ10udG9fbGlzdCgpfSwgZXhwZWN0ZWQge2JbJ3RpY2tlciddLnRvX2xpc3QoKX1cIlxuICAgIGZvciBjIGluIGIuY29sdW1uc1sxOl06XG4gICAgICAgIGZvciB4LCB5IGluIHppcChhW2NdLnRvX2xpc3QoKSwgYltjXS50b19saXN0KCkpOlxuICAgICAgICAgICAgY2xvc2UoeCwgeSwgMWUtNiwgYylcblRFU1RTID0gWyhcImZ1bGwgc2FtcGxlXCIsIGxhbWJkYTogX3QocmV0cykpLCAoXCJoaWRkZW46IDIwMjQgb25seVwiLCBsYW1iZGE6IF90KHJldHMuZmlsdGVyKHBsLmNvbChcImRhdGVcIikuZHQueWVhcigpID09IDIwMjQpKSldXG4iLCAibGFuZyI6ICJweSJ9LCAiZTIiOiB7InNvbHV0aW9uIjogImZyb20gc2tsZWFybi5tb2RlbF9zZWxlY3Rpb24gaW1wb3J0IHRyYWluX3Rlc3Rfc3BsaXRcbmZyb20gc2tsZWFybi5waXBlbGluZSBpbXBvcnQgUGlwZWxpbmVcbmZyb20gc2tsZWFybi5wcmVwcm9jZXNzaW5nIGltcG9ydCBTdGFuZGFyZFNjYWxlclxuZnJvbSBza2xlYXJuLmxpbmVhcl9tb2RlbCBpbXBvcnQgTG9naXN0aWNSZWdyZXNzaW9uXG5mcm9tIHNrbGVhcm4ubWV0cmljcyBpbXBvcnQgcm9jX2F1Y19zY29yZVxuXG5GRUFUVVJFUyA9IFtcImluY29tZVwiLCBcImRlYnRfcmF0aW9cIiwgXCJhZ2VcIiwgXCJsYXRlX3BheW1lbnRzXCJdXG5cbmRlZiBmaXRfZGVmYXVsdChkZik6XG4gICAgWCwgeSA9IGRmLnNlbGVjdChGRUFUVVJFUykudG9fbnVtcHkoKSwgZGZbXCJkZWZhdWx0XCJdLnRvX251bXB5KClcbiAgICBYdHIsIFh0ZSwgeXRyLCB5dGUgPSB0cmFpbl90ZXN0X3NwbGl0KFgsIHksIHRlc3Rfc2l6ZT0wLjMsIHJhbmRvbV9zdGF0ZT0wLCBzdHJhdGlmeT15KVxuICAgIG1vZGVsID0gUGlwZWxpbmUoWyhcInNjYWxlXCIsIFN0YW5kYXJkU2NhbGVyKCkpLCAoXCJsclwiLCBMb2dpc3RpY1JlZ3Jlc3Npb24oKSldKS5maXQoWHRyLCB5dHIpXG4gICAgYXVjID0gcm9jX2F1Y19zY29yZSh5dGUsIG1vZGVsLnByZWRpY3RfcHJvYmEoWHRlKVs6LCAxXSlcbiAgICBjb2VmcyA9IG1vZGVsLm5hbWVkX3N0ZXBzW1wibHJcIl0uY29lZl9bMF1cbiAgICByZXR1cm4ge1wiYXVjXCI6IGZsb2F0KGF1YyksIFwib2Rkc19yYXRpb3NcIjoge2Y6IGZsb2F0KG5wLmV4cChjKSkgZm9yIGYsIGMgaW4gemlwKEZFQVRVUkVTLCBjb2Vmcyl9fVxuIiwgInRlc3RzIjogIlxuZGVmIGNsb3NlKGEsIGIsIHJlbD0xZS02LCB3aGF0PVwidmFsdWVcIik6XG4gICAgYXNzZXJ0IGEgaXMgbm90IE5vbmUsIFwieW91ciBmdW5jdGlvbiByZXR1cm5lZCBOb25lLiBEaWQgeW91IGZvcmdldCB0byByZXR1cm4/XCJcbiAgICBhID0gZmxvYXQoYSk7IGIgPSBmbG9hdChiKVxuICAgIGFzc2VydCBhYnMoYSAtIGIpIDw9IHJlbCAqIG1heCgxLjAsIGFicyhiKSksIGZcInt3aGF0fSA9IHthOi42Z30sIGV4cGVjdGVkIHtiOi42Z31cIlxuZGVmIGJldHdlZW4oYSwgbG8sIGhpLCB3aGF0PVwidmFsdWVcIik6XG4gICAgYXNzZXJ0IGEgaXMgbm90IE5vbmUsIFwieW91ciBmdW5jdGlvbiByZXR1cm5lZCBOb25lLiBEaWQgeW91IGZvcmdldCB0byByZXR1cm4/XCJcbiAgICBhc3NlcnQgbG8gPD0gZmxvYXQoYSkgPD0gaGksIGZcInt3aGF0fSA9IHtmbG9hdChhKTouNGZ9LCBleHBlY3RlZCBiZXR3ZWVuIHtsb30gYW5kIHtoaX1cIlxuZGVmIHNhbWVfa2V5cyhhLCBiLCB3aGF0PVwicmVzdWx0XCIpOlxuICAgIGFzc2VydCBpc2luc3RhbmNlKGEsIGRpY3QpLCBmXCJ7d2hhdH0gc2hvdWxkIGJlIGEgZGljdCwgZ290IHt0eXBlKGEpLl9fbmFtZV9ffVwiXG4gICAgYXNzZXJ0IHNldChhKSA9PSBzZXQoYiksIGZcInt3aGF0fSBrZXlzIHtzb3J0ZWQoYSl9LCBleHBlY3RlZCB7c29ydGVkKGIpfVwiXG5cbmRlZiBfdChkZik6XG4gICAgYSA9IFVbXCJmaXRfZGVmYXVsdFwiXShkZik7IGIgPSBSW1wiZml0X2RlZmF1bHRcIl0oZGYpXG4gICAgc2FtZV9rZXlzKGEsIGIpXG4gICAgY2xvc2UoYVtcImF1Y1wiXSwgYltcImF1Y1wiXSwgMWUtNiwgXCJhdWNcIilcbiAgICBzYW1lX2tleXMoYVtcIm9kZHNfcmF0aW9zXCJdLCBiW1wib2Rkc19yYXRpb3NcIl0sIFwib2Rkc19yYXRpb3NcIilcbiAgICBmb3IgayBpbiBiW1wib2Rkc19yYXRpb3NcIl06XG4gICAgICAgIGNsb3NlKGFbXCJvZGRzX3JhdGlvc1wiXVtrXSwgYltcIm9kZHNfcmF0aW9zXCJdW2tdLCAxZS00LCBcIm9kZHMgcmF0aW8gb2YgXCIgKyBrKVxuVEVTVFMgPSBbKFwiY3JlZGl0XCIsIGxhbWJkYTogX3QoY3JlZGl0KSksIChcImhpZGRlbjogZmlyc3QgMSwyMDAgcm93c1wiLCBsYW1iZGE6IF90KGNyZWRpdC5oZWFkKDEyMDApKSldXG4iLCAibGFuZyI6ICJweSJ9fQ==").decode())
_SETUP = _b64.b64decode("aW1wb3J0IG51bXB5IGFzIG5wCmltcG9ydCBwb2xhcnMgYXMgcGwKaW1wb3J0IG1hdHBsb3RsaWIucHlwbG90IGFzIHBsdApfZyA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZygyMDI2KQpfVCA9IFsiQUFBIiwgIkJCQiIsICJDQ0MiLCAiREREIl0KX2NvdiA9IDAuMDAwMSAqIG5wLmFycmF5KFtbNCwgMiwgMSwgMC41XSwgWzIsIDMsIDEsIDAuNV0sIFsxLCAxLCAyLCAwLjJdLCBbMC41LCAwLjUsIDAuMiwgMV1dKQpfciA9IF9nLm11bHRpdmFyaWF0ZV9ub3JtYWwoWzAuMDAwNiwgMC4wMDA0LCAwLjAwMDMsIDAuMDAwMV0sIF9jb3YsIDUwMCkKX3JbX2cudW5pZm9ybShzaXplPV9yLnNoYXBlKSA8IDAuMDFdID0gbnAubmFuCl9kYXlzID0gbnAuYXJhbmdlKG5wLmRhdGV0aW1lNjQoIjIwMjQtMDEtMDEiKSwgbnAuZGF0ZXRpbWU2NCgiMjAyNi0xMi0zMSIpKQpfZGF5cyA9IF9kYXlzW25wLmlzX2J1c2RheShfZGF5cyldWzo1MDBdCnJldHMgPSBwbC5EYXRhRnJhbWUoeyJkYXRlIjogX2RheXMsICoqe3Q6IF9yWzosIGldIGZvciBpLCB0IGluIGVudW1lcmF0ZShfVCl9fSkud2l0aF9jb2x1bW5zKHBsLmNvbChfVCkuZmlsbF9uYW4oTm9uZSkpCl9tID0gMjAwMApfaW5jID0gbnAucm91bmQoX2cubG9nbm9ybWFsKDExLCAwLjQsIF9tKSwgLTIpCl9kciA9IG5wLnJvdW5kKF9nLmJldGEoMiwgNSwgX20pLCAzKQpfYWdlID0gX2cuaW50ZWdlcnMoMjEsIDcwLCBfbSkKX2xhdGUgPSBfZy5wb2lzc29uKDAuNiwgX20pCl96ID0gLTEuMCAtIDAuMDAwMDIgKiAoX2luYyAtIDYwMDAwKSArIDMuMCAqIF9kciAtIDAuMDIgKiAoX2FnZSAtIDQwKSArIDAuNyAqIF9sYXRlCmNyZWRpdCA9IHBsLkRhdGFGcmFtZSh7ImluY29tZSI6IF9pbmMsICJkZWJ0X3JhdGlvIjogX2RyLCAiYWdlIjogX2FnZSwgImxhdGVfcGF5bWVudHMiOiBfbGF0ZSwKICAgICAgICAgICAgICAgICAgICAgICAiZGVmYXVsdCI6IChfZy51bmlmb3JtKHNpemU9X20pIDwgMSAvICgxICsgbnAuZXhwKC1feikpKS5hc3R5cGUoaW50KX0p").decode()

def _err(e):
    fr = [f for f in _tb.extract_tb(e.__traceback__) if f.filename.startswith("<ipython") or f.filename == "<string>"]
    return f"{type(e).__name__}: {e}"

def check(ex_id, query=None):
    """Run the hidden tests for one exercise, like Submit in the editor."""
    e = _LAB[ex_id]
    ref = {}
    with _ctx.redirect_stdout(_io.StringIO()):
        exec(_SETUP, ref)
        if e["lang"] == "sql":
            ref["__sql__"] = e["solution"]
        else:
            exec(e["solution"], ref)
    if e["lang"] == "sql":
        if query is None:
            print("Pass your query string: check(\"" + ex_id + "\", q)")
            return
        user = {"__sql__": query}
    else:
        user = dict(globals())
    tns = dict(ref)
    tns["U"], tns["R"] = user, ref
    exec(e["tests"], tns)
    ok, tests = 0, tns["TESTS"]
    for name, fn in tests:
        try:
            fn()
            print("  ✓ " + name)
            ok += 1
        except AssertionError as err:
            print("  ✗ " + name + ": " + (str(err) or "wrong answer"))
        except Exception as err:
            print("  ✗ " + name + ": " + _err(err))
    print(("Accepted" if ok == len(tests) else "Wrong answer") + f" · {ok} / {len(tests)} test cases passed")

def solution(ex_id):
    """Print the reference solution. Try once more first."""
    print(_LAB[ex_id]["solution"])

---
## 1. Explore returns with polars and matplotlib
*Medium · id `e1`*

<p><code>rets</code> is a polars DataFrame: a <code>date</code> column and daily returns for four tickers, with a few nulls.</p>
<p>1. Write <code>summary(rets)</code> returning a polars DataFrame with columns <code>ticker, ann_return, ann_vol, sharpe, max_drawdown</code>, one row per ticker, sorted by <code>sharpe</code> (highest first).
<code>ann_return</code> = mean × 252, <code>ann_vol</code> = std × √252 (nulls skipped), <code>sharpe</code> = their ratio, <code>max_drawdown</code> = min of wealth / running peak − 1 where wealth = cumulative product of 1 + r, <b>treating nulls as 0</b> for this column only.</p>
<p>2. Write <code>plot(rets)</code>: cumulative wealth per ticker, and a correlation heatmap (<code>rets.drop("date").corr()</code>). Plots aren't graded.</p>

**Theory to know cold**
<ul><li>polars skips nulls in <code>mean</code> and <code>std</code> (ddof = 1), like pandas.</li><li><code>cum_prod</code>, <code>cum_max</code> are expressions: <code>(1 + pl.col(c).fill_null(0)).cum_prod()</code>.</li><li>Annualising the Sharpe ratio multiplies by √252 because the mean scales with time and the std with its square root.</li></ul>

In [ ]:
def summary(rets: pl.DataFrame) -> pl.DataFrame:
    # your code here
    pass

def plot(rets: pl.DataFrame):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
    # your code here

# Sample run
try:
    print(rets.describe())
    print(summary(rets))
    plot(rets)
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("e1")

---
## 2. Fit and interpret a scikit-learn model
*Medium · id `e2`*

<p><code>credit</code> (polars) has <code>income, debt_ratio, age, late_payments</code> and a 0/1 <code>default</code> column.</p>
<p>Write <code>fit_default(df)</code>: take the features with <code>df.select(FEATURES).to_numpy()</code>, split with <code>train_test_split(test_size=0.3, random_state=0, stratify=y)</code>, fit a <code>Pipeline</code> of <code>StandardScaler</code> then <code>LogisticRegression()</code> (defaults) on the training set, and return a dict with <code>"auc"</code> (test ROC AUC) and <code>"odds_ratios"</code> (feature → exp(coefficient)).</p>
<p>Then explain each odds ratio in one sentence, as you would to an interviewer.</p>

**Theory to know cold**
<ul><li>With scaled inputs, an odds ratio is the multiplicative change in the odds of default per one standard deviation of that feature, holding the others fixed.</li>
<li><code>LogisticRegression</code> is ℓ2-regularised by default (C = 1), so coefficients are shrunk; scikit-learn gives no p-values. For inference use statsmodels <code>Logit</code>.</li>
<li>AUC = probability a random defaulter scores above a random non-defaulter. It ignores calibration; check a reliability curve or the Brier score too.</li>
<li>Fit the scaler inside the pipeline so the test set doesn't leak into it. Stratify when classes are imbalanced. Use cross-validation for model choice, the test set once.</li></ul>

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

FEATURES = ["income", "debt_ratio", "age", "late_payments"]

def fit_default(df: pl.DataFrame) -> dict:
    # your code here
    pass

# Sample run
try:
    print(fit_default(credit))
except Exception as _e:
    print('Sample run failed:', type(_e).__name__, _e)

In [ ]:
check("e2")